# Hyperspy experiments

This is a throwaway notebook just for experimenting. **DO NOT** assume anything in here is correct,
makes sense, or for that matter safe to be in the same room with.

In [ ]:
#import the modules we'll be using later.

import os, re, json, hashlib
from datetime import datetime
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tifffile as tiff
import h5py   # Used for HDF5, if we decide to actually use that.
from scipy import ndimage
from scipy.signal import fftconvolve

import hyperspy.api as hs


# First, the simplest use of HyperSpy just to get the blood flowing and limbs limbered up.

In [ ]:
# straight out of the hyperspy example directory. Possibly with some changes. Likely.

import numpy as np
import hyperspy.api as hs

# Create an image stack with random data
im = hs.signals.Signal2D(np.random.random((16, 32, 32)))

# Define the axis properties
im.axes_manager.signal_axes[0].name = 'X'
im.axes_manager.signal_axes[0].units = 'nm'
im.axes_manager.signal_axes[0].scale = 0.1
im.axes_manager.signal_axes[0].offset = 0

im.axes_manager.signal_axes[1].name = 'Y'
im.axes_manager.signal_axes[1].units = 'nm'
im.axes_manager.signal_axes[1].scale = 0.1
im.axes_manager.signal_axes[1].offset = 0

im.axes_manager.navigation_axes[0].name = 'time'
im.axes_manager.navigation_axes[0].units = 'fs'
im.axes_manager.navigation_axes[0].scale = 0.3
im.axes_manager.navigation_axes[0].offset = 100

# Give a title
im.metadata.General.title = 'Random image stack'
# Plot it
im.plot()


## Now let's define some of the convenience functions we were using in the ChatGPT-generated examples.

In [ ]:
#straight outta Codex...

#
# === Module 0.1 utilities ===



HAS_TIFFFILE = True
HAS_H5PY = True
HAS_SCIPY = True


def read_tiff(path):
    if HAS_TIFFFILE:
        arr = tiff.imread(path)
        return arr
    import matplotlib.image as mpimg
    arr = mpimg.imread(path)
    if arr.ndim == 3:
        arr = arr.mean(axis=2)
    return arr.astype(float)

def save_jpeg_thumbnail(img2d, out_path, p_low=1, p_high=99, max_size=512):
    img = img2d.astype(float)
    lo, hi = np.percentile(img, [p_low, p_high])
    img = np.clip((img - lo) / (hi - lo + 1e-12), 0, 1)
    H, W = img.shape
    scale = max(H, W) / max_size
    if scale > 1:
        step = int(np.ceil(scale))
        img = img[::step, ::step]
    plt.imsave(out_path, img, cmap=None)

def image_stats(img2d):
    img = img2d.astype(float)
    return {
        "shape": list(img.shape),
        "dtype": str(img2d.dtype),
        "min": float(np.min(img)),
        "max": float(np.max(img)),
        "mean": float(np.mean(img)),
        "std": float(np.std(img)),
        "p1": float(np.percentile(img, 1)),
        "p50": float(np.percentile(img, 50)),
        "p99": float(np.percentile(img, 99))
    }

def sha256_file(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def safe_mkdir(p):
    Path(p).mkdir(parents=True, exist_ok=True)

# Go on and turn off traitsui gui on the GUIs tab!!!
#hs.preferences.gui()


## ...and go ahead now and set the paths for inputs and outputs.

In [ ]:
SINGLE_TIFF_PATH = "/Users/escott/projects/working-CITEAM/SiGe/JEM-ARM200FImage_20251210_1036_43_ADF1_ImagePanel1/JEM-ARM200FImage_20251210_1036_46_ADF1_1_ImagePanel1.tif"          # e.g., "data/single/micrograph_01.tif"
STACK_FOLDER_PATH = "/Users/escott/projects/working-CITEAM/SiGe/JEM-ARM200FImage_20251210_1036_43_ADF1_ImagePanel1"         # e.g., "data/stack_01_frames/"
OUTPUT_DIR = "outputs_0_1"
safe_mkdir(OUTPUT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

## For the sake of completeness, let's go ahead and load our single image again. We may decide later we still want to play with it.

In [ ]:
# 0.1.1 — Load a single TIFF
if not SINGLE_TIFF_PATH:
    print("Set SINGLE_TIFF_PATH to a TIFF file.")
else:
    img = read_tiff(SINGLE_TIFF_PATH)
    if img.ndim == 3:
        img = img.mean(axis=2)
    img = img.astype(float)
    print("Single image stats:", json.dumps(image_stats(img), indent=2))
    plt.figure(figsize=(5,5)); plt.imshow(img); plt.title("0.1.1 Single TIFF"); plt.axis("off"); plt.show()
    thumb_path = os.path.join(OUTPUT_DIR, Path(SINGLE_TIFF_PATH).stem + "_thumb.jpg")
    save_jpeg_thumbnail(img, thumb_path)
    print("Saved thumbnail:", thumb_path)

## Load the image again, but this time with HyperSpy. 

**We will create the "s" variable for
playing with HyperSpy or the above-defined "img" variable for doing the Old School way.**

In [ ]:
# hs is hyperspy.


s = hs.load(SINGLE_TIFF_PATH)
print("loaded an image")



In [ ]:
print("s =", s)
print("length is ", len(s))
dir(s[0])

In [ ]:
s[0].plot()

In [ ]:
s[1].plot()

In [ ]:
print(s[0].original_metadata)

In [ ]:
print(s[0].metadata)

In [ ]:
s[0].axes_manager.gui()

## Load an image stack (via HyperSpy's load() function)

**Load a directory full of images, presumably multiple samples of all the same thing,
presumably because we're going to average all of the values together at some point, presumably after alignment.**

**Presumably.**

Python note: the "return sorted(..." line is a very elegant way to step through every filename is a given directory and return an alphabetized list of all the ones that have .tiff or .tif on the end.

In [ ]:
def list_tiff_frames(folder):
    folder = Path(folder)
    exts = {".tif", ".tiff"}
    return sorted([p for p in folder.iterdir() if p.suffix.lower() in exts])

tiffFilesWildcard = STACK_FOLDER_PATH + "/*.tif"
print("tfw = ", tiffFilesWildcard)
#imgStack = hs.load(list_tiff_frames(STACK_FOLDER_PATH))
imgStack = hs.load(tiffFilesWildcard, stack=True)

#print(imgStack[0])
#imgStack[0]

print(type(imgStack))
print(len(imgStack))
#print(imgStack[0])
#print(imgStack[0].metadata)
#print("=================")
#print(imgStack[0].original_metadata)



In [ ]:
wholeStack = hs.load('/Users/escott/projects/working-CITEAM/tentiffs.tif')  # Load stack
print("got 'em")
print(type(wholeStack))
print(wholeStack)
print("metadata =", wholeStack.metadata)
print("original_metadata =", wholeStack.original_metadata)

# ==========================
# This loses the important metadata!
# ==========================

## De-noise the Image


In [ ]:
# it's currently full of 16 bit integers (0 to 65535). We need floating-point values for doing actual math. So we convert:
imgStack[0].change_dtype('float64')
#imgStack[0].decomposition()    # see the plot_* functions from signal.py; you're not done yet!

imgStack[0].plot()



In [ ]:
# save the filtered, shifted image to an HDF5 file

#imgStack[0].save(OUTPUT_DIR + "/savedResult")

# calibrate
from ipywidgets import interact, widgets
%matplotlib widget 
imgStack[0].calibrate(interactive=True)



In [ ]:
imgStack[0].axes_manager.gui()

In [ ]:
# 0.1.1 — Load a TIFF stack from a folder of frames
def list_tiff_frames(folder):
    folder = Path(folder)
    exts = {".tif", ".tiff"}
    return sorted([p for p in folder.iterdir() if p.suffix.lower() in exts])

if not STACK_FOLDER_PATH:
    print("Set STACK_FOLDER_PATH to a folder containing TIFF frames.")
else:
    frame_files = list_tiff_frames(STACK_FOLDER_PATH)
    print("Found frames:", len(frame_files))
    if len(frame_files) == 0:
        raise RuntimeError("No TIFF frames found in STACK_FOLDER_PATH.")

    stack = []
    for p in frame_files:
        f = read_tiff(str(p))
        if f.ndim == 3:
            f = f.mean(axis=2)
        stack.append(f.astype(float))
    stack = np.stack(stack, axis=0)  # (N,H,W)
    print("Stack shape (N,H,W):", stack.shape)

    for i in range(0, len(frame_files)):
        plt.figure(figsize=(5,5)); plt.imshow(stack[i]); plt.title(f"0.1.1 Stack frame {i}"); plt.axis("off"); plt.show()

    plt.figure(figsize=(5,4))
    plt.hist(stack[0][0].ravel(), bins=128)
    plt.title("0.1.2 Histogram (raw)")
    plt.xlabel("Intensity"); plt.ylabel("Count")
    plt.show()

    print("============")
    print("Percentilification :-) ")
    p_low, p_high = 1, 99
    lo, hi = np.percentile(imgStack[0][0], [p_low, p_high])
    img_cs = np.clip((imgStack[0][0] - lo) / (hi - lo + 1e-12), 0, 1)

    gamma = 1.0  # try 0.7 or 1.4
    img_out = np.clip(img_cs ** gamma, 0, 1)

    plt.figure(figsize=(5,5))
    plt.imshow(img_out)
    plt.title(f"0.1.2 Contrast-stretched ({p_low}-{p_high}%) + gamma={gamma}")
    plt.axis("off")
    plt.show()

#    out_jpg = os.path.join(OUTPUT_DIR, Path(SINGLE_TIFF_PATH).stem + f"_cs_{p_low}_{p_high}_g{gamma}.jpg")
#    save_jpeg_thumbnail(img_out, out_jpg, p_low=0, p_high=100)
#    print("Saved JPEG:", out_jpg)

    if HAS_TIFFFILE:
        out_tif = os.path.join(OUTPUT_DIR, Path(SINGLE_TIFF_PATH).stem + f"_cs_{p_low}_{p_high}_g{gamma}.tif")
        img_cs.save(out_tif, overwrite=True)
        print("Saved TIFF:", out_tif)

In [ ]:
if not SINGLE_TIFF_PATH:
    print("Set SINGLE_TIFF_PATH first.")
else:
    img = read_tiff(SINGLE_TIFF_PATH)
    if img.ndim == 3:
        img = img.mean(axis=2)
    img = img.astype(float)

    plt.figure(figsize=(5,4))
    plt.hist(img.ravel(), bins=128)
    plt.title("0.1.2 Histogram (raw)")
    plt.xlabel("Intensity"); plt.ylabel("Count")
    plt.show()

    p_low, p_high = 1, 99
    lo, hi = np.percentile(img, [p_low, p_high])
    img_cs = np.clip((img - lo) / (hi - lo + 1e-12), 0, 1)

    gamma = 1.0  # try 0.7 or 1.4
    img_out = np.clip(img_cs ** gamma, 0, 1)

    plt.figure(figsize=(5,5))
    plt.imshow(img_out)
    plt.title(f"0.1.2 Contrast-stretched ({p_low}-{p_high}%) + gamma={gamma}")
    plt.axis("off")
    plt.show()

# inputs: img_out is the pervcentile-ed, gamma-ed image. Set sigma as needed
# output: dn -s the "de-noised" image.
    sigma = 1.0  # try 0.5, 1.0, 2.0
    dn = ndimage.gaussian_filter(img_out, sigma=sigma)

# now plot the result
    plt.figure(figsize=(5,5))
    plt.imshow(dn)
    plt.title(f"0.1.3 After Gaussian (sigma={sigma})")
    plt.axis("off")
    plt.show()


    out_jpg = os.path.join(OUTPUT_DIR, Path(SINGLE_TIFF_PATH).stem + f"_cs_{p_low}_{p_high}_g{gamma}.jpg")
    save_jpeg_thumbnail(dn, out_jpg, p_low=0, p_high=100)
    print("Saved JPEG:", out_jpg)

    if HAS_TIFFFILE:
        out_tif = os.path.join(OUTPUT_DIR, Path(SINGLE_TIFF_PATH).stem + f"_cs_{p_low}_{p_high}_g{gamma}.tif")
        tiff.imwrite(out_tif, (img_out * 65535).astype(np.uint16))
        print("Saved TIFF:", out_tif)


In [ ]:
    sigma = 1.0  # try 0.5, 1.0, 2.0
    dn = ndimage.gaussian_filter(vis, sigma=sigma)

    plt.figure(figsize=(5,5)); plt.imshow(vis); plt.title("0.1.3 Before"); plt.axis("off"); plt.show()
    plt.figure(figsize=(5,5)); plt.imshow(dn); plt.title(f"0.1.3 After Gaussian (sigma={sigma})"); plt.axis("off"); plt.show()


In [ ]:
if not SINGLE_TIFF_PATH:
    print("Set SINGLE_TIFF_PATH first.")
elif not HAS_SCIPY:
    print("SciPy not available; cannot run gaussian_filter.")
else:
    img = read_tiff(SINGLE_TIFF_PATH)
    if img.ndim == 3:
        img = img.mean(axis=2)
    img = img.astype(float)

    lo, hi = np.percentile(img, [1, 99])
    vis = np.clip((img - lo) / (hi - lo + 1e-12), 0, 1)

    sigma = 1.0  # try 0.5, 1.0, 2.0
    dn = ndimage.gaussian_filter(vis, sigma=sigma)

    plt.figure(figsize=(5,5)); plt.imshow(vis); plt.title("0.1.3 Before"); plt.axis("off"); plt.show()
    plt.figure(figsize=(5,5)); plt.imshow(dn); plt.title(f"0.1.3 After Gaussian (sigma={sigma})"); plt.axis("off"); plt.show()

    out_jpg = os.path.join(OUTPUT_DIR, Path(SINGLE_TIFF_PATH).stem + f"_denoise_sigma{sigma}.jpg")
    save_jpeg_thumbnail(dn, out_jpg, p_low=0, p_high=100)
    print("Saved JPEG:", out_jpg)

    if HAS_TIFFFILE:
        out_tif = os.path.join(OUTPUT_DIR, Path(SINGLE_TIFF_PATH).stem + f"_denoise_sigma{sigma}.tif")
        tiff.imwrite(out_tif, (dn * 65535).astype(np.uint16))
        print("Saved TIFF:", out_tif)

In [ ]:
if not STACK_FOLDER_PATH:
    print("Set STACK_FOLDER_PATH first.")
elif not HAS_SCIPY:
    print("SciPy not available; cannot run drift correction.")
else:
    frame_files = list_tiff_frames(STACK_FOLDER_PATH)
    stack = []
    for p in frame_files:
        f = read_tiff(str(p))
        if f.ndim == 3:
            f = f.mean(axis=2)
        stack.append(f.astype(float))
    stack = np.stack(stack, axis=0)

    # normalize per frame for robust correlation (used ONLY for shift estimation)
    stack_n = np.empty_like(stack)
    for i in range(stack.shape[0]):
        lo, hi = np.percentile(stack[i], [1, 99])
        stack_n[i] = np.clip((stack[i] - lo) / (hi - lo + 1e-12), 0, 1)

    # use average normalized frame as reference for more stable shifts
    ref = stack_n.mean(axis=0)

    def estimate_shift(ref, img):
        a = img - img.mean()
        b = ref - ref.mean()
        corr = fftconvolve(a, b[::-1, ::-1], mode="same")
        y0, x0 = np.unravel_index(np.argmax(corr), corr.shape)
        cy, cx = np.array(corr.shape) // 2
        dy, dx = y0 - cy, x0 - cx
        return -dy, -dx

    aligned = [stack[0]]
    shifts = [(0.0, 0.0)]
    for i in range(1, stack_n.shape[0]):
        dy, dx = estimate_shift(ref, stack_n[i])
        shifts.append((float(dy), float(dx)))
        aligned.append(ndimage.shift(stack[i], shift=(dy, dx), order=1, mode="nearest"))
    aligned = np.stack(aligned, axis=0)

    # compare averages on RAW intensity
    avg_nocorr = stack.mean(axis=0)
    avg_corr = aligned.mean(axis=0)

    print("Shifts (dy, dx):")
    for i, s in enumerate(shifts):
        print(i, s)

    plt.figure(figsize=(5,5)); plt.imshow(avg_nocorr); plt.title("0.1.4 Avg (no correction)"); plt.axis("off"); plt.show()
    plt.figure(figsize=(5,5)); plt.imshow(avg_corr); plt.title("0.1.4 Avg (drift-corrected)"); plt.axis("off"); plt.show()

    print('avg_nocorr type is ', type(avg_nocorr))
    print('avg_nocorr itself is ',avg_nocorr)
    print()
    print('avg_driftcorr type is ', type(avg_nocorr))
    print('avg_driftcorr itself is ',avg_nocorr)
    
    base = Path(STACK_FOLDER_PATH).name.rstrip("/")
    if HAS_TIFFFILE:
        #tiff.imwrite(os.path.join(OUTPUT_DIR, f"{base}_avg_nocorr.tif"), (avg_nocorr*65535).astype(np.uint16))
        #tiff.imwrite(os.path.join(OUTPUT_DIR, f"{base}_avg_driftcorr.tif"), (avg_corr*65535).astype(np.uint16))
        tiff.imwrite(os.path.join(OUTPUT_DIR, f"{base}_avg_nocorr.tif"), avg_nocorr)
        tiff.imwrite(os.path.join(OUTPUT_DIR, f"{base}_avg_driftcorr.tif"), avg_corr)
    save_jpeg_thumbnail(avg_nocorr, os.path.join(OUTPUT_DIR, f"{base}_avg_nocorr.jpg"), p_low=0, p_high=100)
    save_jpeg_thumbnail(avg_corr, os.path.join(OUTPUT_DIR, f"{base}_avg_driftcorr.jpg"), p_low=0, p_high=100)
    print("Saved averaged outputs to", OUTPUT_DIR)

In [ ]:
# 0.1.5 — Interactive backend setup (Jupyter-friendly)
# Tries widget backend; falls back to notebook; otherwise inline + manual input.
try:
    get_ipython
    try:
        %matplotlib widget
        print('Using %matplotlib widget')
    except Exception:
        try:
            %matplotlib notebook
            print('Using %matplotlib notebook')
        except Exception:
            %matplotlib inline
            print('Using %matplotlib inline (manual coordinate fallback will be used)')
except Exception:
    pass

%matplotlib widget


In [ ]:
if not SINGLE_TIFF_PATH:
    print("Set SINGLE_TIFF_PATH first.")
else:
    img = read_tiff(SINGLE_TIFF_PATH)
    if img.ndim == 3:
        img = img.mean(axis=2)
    img = img.astype(float)

    lo, hi = np.percentile(img, [1, 99])
    vis = np.clip((img - lo) / (hi - lo + 1e-12), 0, 1)

    def select_two_points(title, allow_manual=True):
        fig = plt.figure(figsize=(6,6))
        plt.imshow(vis); plt.title(title); plt.axis("off")
        pts = []
        print("going to try to get 2 points - wish me luck!")
        try:
            pts = plt.ginput(2, timeout=0)
        except Exception:
            pts = []
        plt.close(fig)
        
        print("len(pts) = ", len(pts))
        if len(pts) < 2 and allow_manual:
            print("Interactive selection failed. Enter coordinates manually as x1,y1,x2,y2")
            raw = input("Coords: ").strip()
            parts = [p.strip() for p in raw.split(",")]
            if len(parts) == 4:
                x1, y1, x2, y2 = map(float, parts)
                pts = [(x1, y1), (x2, y2)]
        return pts

    print("Click TWO points for the feature (line profile), then press Enter.")
    pts = select_two_points("Feature: click 2 points, Enter")
    if len(pts) < 2:
        raise RuntimeError("No points captured. Ensure interactive backend or use manual input.")

    (x1, y1), (x2, y2) = pts
    feature_px = float(np.hypot(x2-x1, y2-y1))
    print(f"Feature length: {feature_px:.2f} px")

    # --- Line profile along the selected feature ---
    n_samples = int(max(2, np.ceil(feature_px)))
    xs = np.linspace(x1, x2, n_samples)
    ys = np.linspace(y1, y2, n_samples)

    try:
        # use scipy if available
        if HAS_SCIPY:
            from scipy.ndimage import map_coordinates
            profile = map_coordinates(img, [ys, xs], order=1, mode='nearest')
        else:
            # nearest-neighbor fallback
            xi = np.clip(np.round(xs).astype(int), 0, img.shape[1]-1)
            yi = np.clip(np.round(ys).astype(int), 0, img.shape[0]-1)
            profile = img[yi, xi]
    except Exception:
        xi = np.clip(np.round(xs).astype(int), 0, img.shape[1]-1)
        yi = np.clip(np.round(ys).astype(int), 0, img.shape[0]-1)
        profile = img[yi, xi]

    plt.figure(figsize=(6,3))
    plt.plot(profile)
    plt.title("0.1.5 Line profile (intensity vs. distance)")
    plt.xlabel("Sample index")
    plt.ylabel("Intensity")
    plt.show()

    # Save profile to CSV
    prof_path = os.path.join(OUTPUT_DIR, Path(SINGLE_TIFF_PATH).stem + "_line_profile.csv")
    with open(prof_path, "w", encoding="utf-8") as f:
        f.write("index,intensity")
        for k, v in enumerate(profile):
            f.write(f"{k},{float(v)}")
    print("Saved line profile:", prof_path)

    print("Now click TWO endpoints of the scale bar, then press Enter.")
    sb = select_two_points("Scale bar: click 2 points, Enter")
    if len(sb) < 2:
        raise RuntimeError("No scale-bar points captured.")
    (sx1, sy1), (sx2, sy2) = sb
    sb_px = float(np.hypot(sx2-sx1, sy2-sy1))
    print(f"Scale bar length: {sb_px:.2f} px")

    sb_nm = float(input("Enter the scale bar length in nm (read from label): "))
    nm_per_px = sb_nm / max(sb_px, 1e-12)
    feature_nm = feature_px * nm_per_px
    print(f"Calibration: {nm_per_px:.6f} nm/px")
    print(f"Feature length: {feature_nm:.2f} nm")

    out = {
        "input_file": SINGLE_TIFF_PATH,
        "feature_px": feature_px,
        "scale_bar_px": sb_px,
        "scale_bar_nm": sb_nm,
        "nm_per_px": nm_per_px,
        "feature_nm": feature_nm,
        "created_utc": datetime.utcnow().isoformat() + "Z",
        "line_profile_csv": prof_path,
    }
    out_path = os.path.join(OUTPUT_DIR, Path(SINGLE_TIFF_PATH).stem + "_measurement.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(out, f, indent=2)
    print("Saved measurement JSON:", out_path)

In [ ]:

if not SINGLE_TIFF_PATH:
    print("Set SINGLE_TIFF_PATH first.")
else:
    img = read_tiff(SINGLE_TIFF_PATH)
    if img.ndim == 3:
        img = img.mean(axis=2)
    img = img.astype(float)

    p_low, p_high = 1, 99
    lo, hi = np.percentile(img, [p_low, p_high])
    proc = np.clip((img - lo) / (hi - lo + 1e-12), 0, 1)
    sigma = 1.0
    if HAS_SCIPY:
        proc = ndimage.gaussian_filter(proc, sigma=sigma)

    in_sha = sha256_file(SINGLE_TIFF_PATH) if os.path.exists(SINGLE_TIFF_PATH) else None
    meta = {
        "input_file": SINGLE_TIFF_PATH,
        "input_sha256": in_sha,
        "created_utc": datetime.utcnow().isoformat() + "Z",
        "processing": [
            {"step": "percentile_contrast", "p_low": p_low, "p_high": p_high},
            {"step": "gaussian_denoise", "sigma": sigma, "scipy": bool(HAS_SCIPY)},
        ],
        "stats_raw": image_stats(img),
        "stats_processed": image_stats(proc),
        "software": {
            "python": f"{os.sys.version_info.major}.{os.sys.version_info.minor}.{os.sys.version_info.micro}",
            "numpy": np.__version__,
        },
    }

    base = Path(SINGLE_TIFF_PATH).stem
    h5_path = os.path.join(OUTPUT_DIR, base + "_bundle.h5")
    readme_path = os.path.join(OUTPUT_DIR, base + "_README.txt")

    if HAS_H5PY:
        import h5py
        with h5py.File(h5_path, "w") as f:
            f.create_dataset("raw", data=img, compression="gzip")
            f.create_dataset("processed", data=proc, compression="gzip")
            f.attrs["metadata_json"] = json.dumps(meta)
        print("Wrote HDF5:", h5_path)
    else:
        print("h5py not available; skipping HDF5 export.")

    readme = f"""TEM Processing README
====================
Input file: {SINGLE_TIFF_PATH}
Input sha256: {in_sha}
Created (UTC): {meta['created_utc']}

Processing:
- Percentile contrast: {p_low}–{p_high}%
- Gaussian denoise: sigma={sigma} (SciPy: {bool(HAS_SCIPY)})

Raw stats: {meta['stats_raw']}
Processed stats: {meta['stats_processed']}

Software:
- Python: {meta['software']['python']}
- NumPy: {meta['software']['numpy']}
"""

    with open(readme_path, "w", encoding="utf-8") as f:
        f.write(readme)
        f.write("\nCode excerpt:\n")
        f.write("  lo, hi = np.percentile(img, [p_low, p_high])\n")
        f.write("  proc = np.clip((img - lo) / (hi - lo + 1e-12), 0, 1)\n")
        f.write("  proc = ndimage.gaussian_filter(proc, sigma=sigma)\n")

    print("Wrote README:", readme_path)
